# CoT-NAFNet on CDD-11-30

Import this notebook from GitHub, attach the two existing Kaggle inputs, enable a GPU, and run all cells. The default run performs only a read-only audit. Set `RUN_TRAIN = True` after the audit succeeds.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("CWD:", Path.cwd())

In [ ]:
from pathlib import Path
import torch

CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
assert CDD11_ROOT.is_dir(), f"Missing CDD-11 input: {CDD11_ROOT}"
assert PRETRAINED_ROOT.is_dir(), f"Missing pretrained input: {PRETRAINED_ROOT}"
assert torch.cuda.is_available(), "Enable a Kaggle GPU before continuing"
print("GPU:", torch.cuda.get_device_name(0))
print("CDD-11:", CDD11_ROOT)
print("Pretrained:", sorted(path.name for path in PRETRAINED_ROOT.glob("*.pth")))

In [ ]:
# Read-only audit: all pairs, split leakage, and exact compatibility of all four checkpoints.
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT),
    "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/cot_nafnet_audit/audit.json",
], check=True)

In [ ]:
# Safety switches. A freshly imported notebook audits inputs but does not start a long run.
RUN_PRETRAINED_PROBE = False  # About 4-10 minutes for all four models on validation.
PROBE_PRESETS = ["gopro32", "gopro64", "sidd32", "sidd64"]
RUN_QUALITATIVE_PROBE = False  # About 1-3 minutes for SIDD32.
QUALITATIVE_PRESET = "sidd32"
RUN_TRAIN = False
RUN_EVALUATION = False
MODEL = "baseline"  # Five-epoch timing/convergence calibration.
PRESET = "sidd32"  # Selected provisionally by the validation probe.
USE_SKIP_GATES = False
DEGRADATION_WEIGHT = 0.0
CONTENT_WEIGHT = 0.0
DECORRELATION_WEIGHT = 0.0
GATE_WEIGHT = 0.0
FREEZE_BACKBONE_EPOCHS = 0
EPOCHS = 5
OUTPUT_DIR = Path("/kaggle/working/experiments/calibration_baseline_sidd32_seed42_ramfix")

In [ ]:
if RUN_PRETRAINED_PROBE:
    subprocess.run([
        "python", "-m", "hybrid_cot_nafnet.probe_pretrained_cdd11",
        "--data-root", str(CDD11_ROOT),
        "--pretrained-root", str(PRETRAINED_ROOT),
        "--output-dir", "/kaggle/working/pretrained_cdd11_validation_probe",
        "--split", "validation",
        "--presets", *PROBE_PRESETS,
        "--tile", "256", "--overlap", "32",
    ], check=True)
else:
    print("Pretrained CDD-11 probe skipped.")

if RUN_QUALITATIVE_PROBE:
    subprocess.run([
        "python", "-m", "hybrid_cot_nafnet.probe_pretrained_cdd11",
        "--data-root", str(CDD11_ROOT),
        "--pretrained-root", str(PRETRAINED_ROOT),
        "--output-dir", "/kaggle/working/pretrained_cdd11_qualitative",
        "--split", "validation",
        "--presets", QUALITATIVE_PRESET,
        "--save-comparisons", "--max-saved-per-type", "1",
        "--tile", "256", "--overlap", "32",
    ], check=True)
else:
    print("Qualitative probe skipped.")

In [ ]:
if RUN_TRAIN:
    command = [
        "python", "-m", "hybrid_cot_nafnet.train_kaggle",
        "--data-root", str(CDD11_ROOT),
        "--output-dir", str(OUTPUT_DIR),
        "--model", MODEL,
        "--preset", PRESET,
        "--pretrained", "auto",
        "--multi-gpu",
        "--skip-gates" if USE_SKIP_GATES else "--no-skip-gates",
        "--degradation-weight", str(DEGRADATION_WEIGHT),
        "--content-weight", str(CONTENT_WEIGHT),
        "--decorrelation-weight", str(DECORRELATION_WEIGHT),
        "--gate-weight", str(GATE_WEIGHT),
        "--freeze-backbone-epochs", str(FREEZE_BACKBONE_EPOCHS),
        "--epochs", str(EPOCHS),
        "--max-minutes", "0",
        "--crop-size", "256",
        "--batch-size", "4",
        "--microbatch-size", "2",
        "--patches-per-image", "2",
        "--num-workers", "0",
        "--no-pin-memory",
        "--save-every", "0",
        "--no-save-optimizer",
        "--seed", "42",
    ]
    subprocess.run(command, check=True)
else:
    print("Training skipped. Set RUN_TRAIN = True after reviewing audit.json.")

In [ ]:
if RUN_EVALUATION:
    checkpoint = OUTPUT_DIR / "best.pt"
    assert checkpoint.is_file(), f"Missing checkpoint: {checkpoint}"
    subprocess.run([
        "python", "-m", "hybrid_cot_nafnet.evaluate",
        "--checkpoint", str(checkpoint),
        "--data-root", str(CDD11_ROOT),
        "--output-dir", str(OUTPUT_DIR / "evaluation"),
        "--split", "validation",
        "--tile", "256", "--overlap", "32",
        "--num-workers", "2",
    ], check=True)
else:
    print("Evaluation skipped. Set RUN_EVALUATION = True after training.")

In [ ]:
# Show generated files without searching manually in the Files panel.
from IPython.display import FileLinks, display
for artifact_dir in [
    Path("/kaggle/working/cot_nafnet_audit"),
    Path("/kaggle/working/pretrained_cdd11_validation_probe"),
    Path("/kaggle/working/pretrained_cdd11_qualitative"),
    OUTPUT_DIR,
]:
    if artifact_dir.is_dir():
        print(f"Artifacts in {artifact_dir}:")
        display(FileLinks(str(artifact_dir)))